In [18]:
import os
import cv2
import numpy as np
import pandas as pd


# 1. دالة استخراج الهستغرام اللوني
def extract_color_histogram(image, bins=(8, 8, 8)):
  hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
  hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
  cv2.normalize(hist, hist)
  return hist.flatten()


data = []
labels = []
rows_for_excel = []

# 2. تحديد مسار المجلد الرئيسي مباشرة (بدون تكرار كلمة dataset)
dataset_path = r"C:\Users\Ramadan\Downloads\Dr.1\Dr.1\dataset"
output_dir = r"C:\Users\Ramadan\Downloads\Dr.1\Dr.1"

categories = {"cats": 0, "dogs": 1}

print(f"جاري البحث عن الصور في المسار: ")

# 3. التكرار عبر الصور
for category, label in categories.items():
  folder_path = os.path.join(dataset_path, category)

  if not os.path.exists(folder_path):
    print(f"⚠️ تحذير: المجلد غير موجود -> ")
    continue

  count = 0
  for img_name in os.listdir(folder_path):
    if not img_name.lower().endswith(VALID_EXTENSIONS):
      continue

    img_path = os.path.join(folder_path, img_name)
    image = cv2.imread(img_path)

    if image is None:
      print(f"⚠️ تعذر قراءة الصورة: ")
      continue

    image = cv2.resize(image, (128, 128))
    features = extract_color_histogram(image)

    data.append(features)
    labels.append(label)
    count += 1

    # تجهيز سجل مفصل لملف الإكسل
    row = {"Image_Name": img_name, "Category": category, "Label": label}
    for idx, hist_val in enumerate(features):
      row[f"Hist_Bin_{idx+1}"] = round(hist_val, 5)
    rows_for_excel.append(row)

  print(f" تم تحميل {count} صورة من صنف '{category}'")

# 4. حفظ البيانات في ملف npz وملف Excel
if len(data) == 0:
  print(
      "\n❌ خطأ: لم يتم العثور على أي صورة! تأكد من وجود الصور داخل مجلدات"
      " cats و dogs."
  )
else:
  # أ) حفظ ملف NumPy (.npz)
  X = np.array(data)
  y = np.array(labels)

  npz_output_file = os.path.join(output_dir, "processed_data.npz")
  np.savez(npz_output_file, X=X, y=y)
  print(f"\n✅ تم استخراج السمات بنجاح! إجمالي الصور: {len(X)}")
  print(f"تم حفظ ملف NPZ في: ")

  # ب) حفظ ملف Excel (.xlsx)
  df = pd.DataFrame(rows_for_excel)
  excel_path = os.path.join(output_dir, "extracted_features.xlsx")
  df.to_excel(excel_path, index=False, engine="openpyxl")
  print(f"✅ تم حفظ ملف الإكسل بنجاح في: ")

جاري البحث عن الصور في المسار: 
 تم تحميل 30 صورة من صنف 'cats'
⚠️ تعذر قراءة الصورة: 
 تم تحميل 28 صورة من صنف 'dogs'

✅ تم استخراج السمات بنجاح! إجمالي الصور: 58
تم حفظ ملف NPZ في: 
✅ تم حفظ ملف الإكسل بنجاح في: 


In [13]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. قراءة البيانات من ملف Excel
df = pd.read_excel('extracted_features.xlsx')

# 2. فصل سمات الصور (X) والتصنيف (y)
# استبعاد أعمدة اسم الصورة والتصنيف النصي
X = df.drop(columns=['Image_Name', 'Category', 'Label']).values
y = df['Label'].values

# 3. تقسيم البيانات لتدريب واختبار
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. المعايرة المعيارية
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. تدريب ومقارنة الخوارزميات
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Support Vector Machine (SVM)': SVC(kernel='rbf', random_state=42),
    'K-Nearest Neighbors (KNN)': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    results.append({
        'Algorithm': name,
        'Accuracy': f"{accuracy_score(y_test, y_pred):.2%}",
        'Precision': f"{precision_score(y_test, y_pred, zero_division=0):.2%}",
        'Recall': f"{recall_score(y_test, y_pred, zero_division=0):.2%}",
        'F1-Score': f"{f1_score(y_test, y_pred, zero_division=0):.2%}"
    })

# 6. عرض النتائج
df_results = pd.DataFrame(results)
print("=== نتائج المقارنة بناءً على سمات ملف Excel ===")
print(df_results.to_string(index=False))

=== نتائج المقارنة بناءً على سمات ملف Excel ===
                   Algorithm Accuracy Precision  Recall F1-Score
               Random Forest   50.00%    50.00%  33.33%   40.00%
Support Vector Machine (SVM)   50.00%    50.00% 100.00%   66.67%
   K-Nearest Neighbors (KNN)   50.00%    50.00%  33.33%   40.00%
               Decision Tree   50.00%     0.00%   0.00%    0.00%


In [15]:
import cv2
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# =========================================================
# 1. قراءة البيانات التدريبية وتقسيمها
# =========================================================
excel_file = "extracted_features.xlsx"

try:
  print("📂 جاري قراءة ملف الإكسل...")
  df = pd.read_excel(excel_file)
  print(f"✅ تم تحميل الملف بنجاح. أبعاد البيانات: {df.shape}")
except Exception as e:
  print(f"❌ لم يتم العثور على ملف الإكسل: {excel_file}")
  exit()

# استبعاد الأعمدة غير الرقمية
X = df.drop(columns=["Image_Name", "Category", "Label"], errors="ignore").values
y = df["Label"].values

print(
    f"📊 عدد العينات (الصور): {X.shape[0]} | عدد السمات الأصلي: {X.shape[1]}"
)

# تقسيم البيانات لـ Train و Test لحساب مقاييس الأداء (Accuracy, Precision, Recall, F1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# تقييم وتدريب التقييس و PCA على مجموعة التدريب فقط لمنع تسريب البيانات
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

n_components = min(10, X_train.shape[0] - 1)
pca = PCA(n_components=n_components, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

explained_variance = np.sum(pca.explained_variance_ratio_) * 100
print(
    f"🚀 تم تقليل السمات إلى {n_components} سمات فقط! (تغطي {explained_variance:.1f}% من معلومات البيانات)"
)

# =========================================================
# 2. إعداد الخوارزميات واحتساب تقييم الأداء (Metrics)
# =========================================================
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=3, random_state=42
    ),
    "SVM": SVC(kernel="rbf", C=1.0, probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "QDA": QuadraticDiscriminantAnalysis(solver="eigen", shrinkage="auto"),
}

results = []
trained_models = {}

print("\n⚙️ جاري تدريب الخوارزميات وتقييم الأداء...")
for name, model in models.items():
  try:
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)

    results.append({
        "Algorithm": name,
        "Accuracy": f"{accuracy_score(y_test, y_pred):.2%}",
        "Precision": f"{precision_score(y_test, y_pred, zero_division=0):.2%}",
        "Recall": f"{recall_score(y_test, y_pred, zero_division=0):.2%}",
        "F1-Score": f"{f1_score(y_test, y_pred, zero_division=0):.2%}",
    })
    trained_models[name] = model
  except Exception as e:
    print(f" ❌ فشل تدريب {name}: {e}")

# عرض جدول التقييم في Terminal
results_df = pd.DataFrame(results)
print("\n📊 جدول تقييم الخوارزميات على بيانات الاختبار (Test Split):")
print(results_df.to_string(index=False))


# =========================================================
# 3. دالة استخراج سمات الصورة
# =========================================================
def extract_single_image_features(image):
  mean_b, mean_g, mean_r = cv2.mean(image)[:3]
  std_b, std_g, std_r = cv2.meanStdDev(image)[1].flatten()[:3]

  hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
  hist = cv2.calcHist([hsv], [0, 1, 2], None, (8, 8, 8), [0, 180, 0, 256, 0, 256])
  cv2.normalize(hist, hist)

  feats = [mean_b, mean_g, mean_r, std_b, std_g, std_r] + list(hist.flatten())
  return np.array(feats).reshape(1, -1)


# =========================================================
# 4. التنبؤ وتجميع كافة النتائج أسفل الصورة
# =========================================================
test_image_path = r"C:\Users\Ramadan\Downloads\Dr.1\Dr.1\test\cat.jpeg"
original_img = cv2.imread(test_image_path)

if original_img is None:
  print(f"\n❌ تعذر العثور على صورة الاختبار: {test_image_path}")
else:
  feature_img = cv2.resize(original_img, (128, 128))
  raw_features = extract_single_image_features(feature_img)

  if raw_features.shape[1] != X.shape[1]:
    print(f"\n❌ خطأ: عدم مطابقة السمات مع الإكسل!")
    exit()

  scaled_features = scaler.transform(raw_features)
  pca_features = pca.transform(scaled_features)

  labels_map = {1: "Cat (قطة)", 0: "Dog (كلب)"}

  # تحضير صورة التست وعرض النتائج أسفلها مباشرة
  img_resized = cv2.resize(original_img, (450, 350))
  row_height = 35
  panel_height = len(trained_models) * row_height + 40
  panel = (
      np.zeros((panel_height, 450, 3), dtype=np.uint8) + 30
  )  # لوحة سوداء للنتائج

  cv2.putText(
      panel,
      "Algorithm Results Comparison",
      (15, 25),
      cv2.FONT_HERSHEY_SIMPLEX,
      0.6,
      (0, 255, 255),
      2,
  )

  y_offset = 60
  print("\n🔍 نتائج التنبؤ لصورة الاختبار:")
  for name, model in trained_models.items():
    pred = model.predict(pca_features)[0]
    result_text = labels_map.get(pred, str(pred))

    if hasattr(model, "predict_proba"):
      prob = model.predict_proba(pca_features)[0][pred] * 100
      confidence_text = f"{prob:.1f}%"
    else:
      confidence_text = "N/A"

    line = f"{name}:  ({confidence_text})"
    print(f" • {line}")

    cv2.putText(
        panel,
        line,
        (15, y_offset),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        1,
    )
    y_offset += row_height

  # تجميع الصورة الأصلية مع اللوحة السفلية
  combined_view = np.vstack((img_resized, panel))

  cv2.imshow("All Models Prediction Results", combined_view)
  cv2.waitKey(0)
  cv2.destroyAllWindows()

📂 جاري قراءة ملف الإكسل...
✅ تم تحميل الملف بنجاح. أبعاد البيانات: (28, 521)
📊 عدد العينات (الصور): 28 | عدد السمات الأصلي: 518
🚀 تم تقليل السمات إلى 10 سمات فقط! (تغطي 83.2% من معلومات البيانات)

⚙️ جاري تدريب الخوارزميات وتقييم الأداء...


C:\Users\Ramadan\anaconda3\envs\Maryam\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



📊 جدول تقييم الخوارزميات على بيانات الاختبار (Test Split):
    Algorithm Accuracy Precision Recall F1-Score
Random Forest   50.00%    50.00% 66.67%   57.14%
          SVM   66.67%    66.67% 66.67%   66.67%
          KNN   66.67%    66.67% 66.67%   66.67%
Decision Tree   50.00%     0.00%  0.00%    0.00%
          QDA   50.00%    50.00% 66.67%   57.14%

🔍 نتائج التنبؤ لصورة الاختبار:
 • Random Forest:  (54.0%)
 • SVM:  (69.8%)
 • KNN:  (66.7%)
 • Decision Tree:  (100.0%)
 • QDA:  (100.0%)
